In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [ ]:
# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python Challenge/compile_cython.py
os.chdir(WORKING_DIR)

In [ ]:
%%capture

!pip install optuna
import optuna

In [ ]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

In [ ]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

## **Explore URM to segment them in differnt categories**

# ideas:
- Sparsity (number of interaction)
- Similarity (some similarity)
- Percentage of interactions with rare items

Instead of segmenting with some heuristic we can separate the user based on which model perform better on them

In [ ]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Load Models**

In [ ]:
models = {}

### **SLIM**

In [ ]:
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

# Load SLIM model
slim_model = SLIMElasticNetRecommender(URM_train)
slim_model.load_model(paths.MODEL_DIR)

# Add SLIM model to models dictionary
models['SLIM'] = slim_model

# Evaluate SLIM model
slim_recall = evaluate_recommender(slim_model, at=20)
print(f"SLIM Model - Recall@20: {slim_recall:.5f}")

### **IALS**

In [ ]:
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

# Load IALS model
ials_model = IALSRecommender(URM_train)
ials_model.load_model(paths.MODEL_DIR)

# Add IALS model to models dictionary
models['IALS'] = ials_model

# Evaluate IALS model
ials_recall = evaluate_recommender(ials_model, at=20)
print(f"IALS Model - Recall@20: {ials_recall:.5f}")

### **TopPop**

In [ ]:
from Recommenders.NonPersonalizedRecommender import TopPop

# Train TopPop model
toppop_model = TopPop(URM_train)
toppop_model.fit()

# Add TopPop model to models dictionary
models['TopPop'] = toppop_model

# Evaluate TopPop model
toppop_recall = evaluate_recommender(toppop_model, at=20)
print(f"TopPop Model - Recall@20: {toppop_recall:.5f}")

### **KNN**

In [ ]:
SIMILIARITIES = ["cosine", "pearson", "jaccard", "tversky"]

#### **UserKNN**

In [ ]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
KNN_MODEL_DIR = os.path.join(paths.MODEL_DIR, "UserKNN")

for sim in SIMILIARITIES:
    # Load UserKNN model
    userknn_model = UserKNNCFRecommender(URM_train)
    userknn_model.load_model(KNN_MODEL_DIR, file_name=sim)

    # Add UserKNN model to models dictionary
    models['UserKNN'+sim] = userknn_model

    # Evaluate UserKNN model
    userknn_recall = evaluate_recommender(userknn_model, at=20)
    print(f"UserKNN {sim} Model - Recall@20: {userknn_recall:.5f}")

#### **ItemKNN**

In [ ]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
KNN_MODEL_DIR = os.path.join(paths.MODEL_DIR, "ItemKNN")

for sim in SIMILIARITIES:
    # Load ItemKNN model
    itemknn_model = ItemKNNCFRecommender(URM_train)
    itemknn_model.load_model(KNN_MODEL_DIR, file_name=sim)

    # Add ItemKNN model to models dictionary
    models['ItemKNN'+sim] = itemknn_model

    # Evaluate ItemKNN model
    itemknn_recall = evaluate_recommender(itemknn_model, at=20)
    print(f"ItemKNN {sim} Model - Recall@20: {itemknn_recall:.5f}")

## **Evaluate performance of each model per user**

In [ ]:
df_models_performance = pd.DataFrame(index=np.arange(URM_train.shape[0]), columns=models.keys())

# Populate performance DataFrame
for user_id in range(URM_validation.shape[0]):
    relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
    
    if len(relevant_items)>0:
        for model_name, model in models.items():
            recommended_items = model.recommend(user_id, cutoff=20)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            # Store recall score in DataFrame
            df_models_performance.at[user_id, model_name] = recall_score

# Save performance DataFrame to CSV
df_models_performance.to_csv(os.path.join(paths.RESULTS_DIR, "model_performance_per_user.csv"), index_label="user_id")

In [ ]:
# Print recall for each model to verify
for model_name in models.keys():
    recall = df_models_performance[model_name].mean()
    print(f"Model: {model_name}, Recall@20: {recall:.5f}")

In [ ]:
# Count best model per user
best_model_counts = df_models_performance.idxmax(axis=1).value_counts()
print("Best model counts per user:")
print(best_model_counts)

In [ ]:
# 1. Get each user's maximum performance
max_per_user = df_models_performance.max(axis=1)

# 2. Boolean mask of which models hit that max
is_best = df_models_performance.eq(max_per_user, axis=0)

# 3. For each user, extract tuple of best models (sorted for consistent grouping)
best_combos = is_best.apply(
    lambda row: tuple(sorted(row.index[row].tolist())), axis=1
)

# 4. Count occurrences of each unique combination
combo_counts = best_combos.value_counts().rename_axis("best_models_combo").reset_index(name="n_users")

print(combo_counts.head(15))

In [ ]:
import matplotlib.pyplot as plt

combo_counts.head(15).plot.barh(
    x="best_models_combo", y="n_users", figsize=(8,5), legend=False
)
plt.xlabel("Number of users")
plt.ylabel("Model combination (tied best)")
plt.title("Users per best-model combination")
plt.gca().invert_yaxis()
plt.show()
